# Making Predictions with Linear Regression

## 1. Introduction and Model Recap

In the previous notebook, we learned how to fit a Simple Linear Regression (SLR) model to understand the relationship between two variables. 

**The Model Equation:**

$$y = \beta_0 + \beta_1 x + \epsilon$$

Once we have estimated the coefficients ($\beta_0$ and $\beta_1$), we can use the model to make **predictions** for new observations. This is one of the most powerful applications of regression analysis.

In this notebook, we will cover:
1.  The Prediction Equation.
2.  Interpolation vs. Extrapolation.
3.  Calculatiing Predictions with `statsmodels`.
4.  Confidence and Prediction Intervals.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Set plot style
sns.set_theme(style="whitegrid")

In [ ]:
# Load the dataset and fit the model (Recap from C1)
file_path = "../../00-data/Introduction to Regression Fish.csv"
df = pd.read_csv(file_path)

X = df['length_cm']
y = df['mass_g']
X_with_const = sm.add_constant(X)
model = sm.OLS(y, X_with_const).fit()

print("Model Fitted Successfully")
print(model.params)

## 2. The Prediction Equation

To make a prediction, we simply plug a new value of $x$ into our fitted equation:

$$ \hat{y} = \hat{\beta}_0 + \hat{\beta}_1 x_{new} $$

Where:
*   $\hat{y}$ is the predicted value.
*   $\hat{\beta}_0$ is the estimated intercept.
*   $\hat{\beta}_1$ is the estimated slope.
*   $x_{new}$ is the new input value.

**Example:**
If our Intercept is -500 and our Slope is 25, and we want to predict the mass of a fish that is 30 cm long:

$$ \text{Mass} = -500 + 25(30) = -500 + 750 = 250\text{ g} $$

## 3. Interpolation vs. Extrapolation

When making predictions, it is crucial to check whether the new $x$ value lies within the range of data used to train the model.

### **Interpolation**
Making predictions **inside** the range of the original data. 
*   **Reliability:** High. We have evidence of the relationship in this region.
*   *Example: Predicting for a fish of length 30cm, when our data has fish from 10cm to 50cm.*

### **Extrapolation**
Making predictions **outside** the range of the original data.
*   **Reliability:** Low (Risky). We assume the linear relationship continues indefinitely, which is often not true (e.g., biological limits).
*   *Example: Predicting for a fish of length 200cm. A linear model might predict a massive weight, but in reality, the fish might not exist or the relationship might curve.*

![Extrapolation vs Interpolation](https://www.statology.org/wp-content/uploads/2021/09/interp3.png)
*(Image Source: Wikimedia Commons. The red dots are known data, blue is the interpolated line, and the dotted line represents extrapolation.)*

In [ ]:
# Visualizing Data Range
print(f"Minimum Length in Data: {df['length_cm'].min()} cm")
print(f"Maximum Length in Data: {df['length_cm'].max()} cm")

## 4. Coding Predictions (statsmodels)

We use the `.predict()` method. Note that we must provide the data in the same format as the training data, including the constant term!

In [ ]:
# 1. Create a DataFrame for the new values
new_lengths = pd.DataFrame({'length_cm': [25, 30, 35, 100]})

# 2. Add the constant column 
new_lengths_with_const = sm.add_constant(new_lengths, has_constant='add')

# 3. Predict
predictions = model.predict(new_lengths_with_const)

# Add predictions to the DataFrame for viewing
new_lengths['predicted_mass'] = predictions
new_lengths['type'] = ['Interpolation', 'Interpolation', 'Interpolation', 'Extrapolation']
print(new_lengths)

## 5. Confidence and Prediction Intervals

A single point prediction (like 250g) doesn't tell us about the uncertainty. 

### **Confidence Interval (CI)**
*   Estimates the **mean** value of $y$ for a given $x$.
*   *Question: "What is the average mass of all fish that are 30cm long?"*
*   Narrower range.

### **Prediction Interval (PI)**
*   Estimates the value of a **single individual** observation $y$ for a given $x$.
*   *Question: "What is the mass of this specific 30cm fish I just caught?"*
*   Wider range (accounts for the variability of individual data points around the mean).

![CI vs PI](https://www.graphpad.com/guides/prism/latest/curve-fitting/images/hm_graphing_confidence_vs_prediction_bands.png)
*(Concept Image: The darker shaded area determines the Confidence Interval, while the lighter, wider area is the Prediction Interval.) *

In [ ]:
# Calculating Intervals using get_prediction()
pred_results = model.get_prediction(new_lengths_with_const)

# Summary frame contains mean, mean_se, mean_ci_lower, mean_ci_upper, obs_ci_lower, obs_ci_upper
summary_frame = pred_results.summary_frame(alpha=0.05) # 95% confidence
print(summary_frame.head())

In [ ]:
# Visualizing the Intervals
# Let's generate a range of X values for plotting
x_range = np.linspace(df['length_cm'].min(), df['length_cm'].max(), 100)
x_range_df = pd.DataFrame({'length_cm': x_range})
x_range_const = sm.add_constant(x_range_df)

predictions = model.get_prediction(x_range_const)
pred_summary = predictions.summary_frame()

plt.figure(figsize=(10, 6))

# Plot Data
sns.scatterplot(x='length_cm', y='mass_g', data=df, alpha=0.5, label='Observed Data')

# Plot Mean Prediction (Regression Line)
plt.plot(x_range, pred_summary['mean'], color='red', label='Regression Line')

# Plot Confidence Interval (Mean CI)
plt.fill_between(x_range, pred_summary['mean_ci_lower'], pred_summary['mean_ci_upper'], color='blue', alpha=0.2, label='95% Confidence Interval')

# Plot Prediction Interval (Obs CI)
plt.fill_between(x_range, pred_summary['obs_ci_lower'], pred_summary['obs_ci_upper'], color='green', alpha=0.1, label='95% Prediction Interval')

plt.title('Confidence vs. Prediction Intervals')
plt.xlabel('Length (cm)')
plt.ylabel('Mass (g)')
plt.legend()
plt.show()

## 6. Exercises

**Exercise 1:** Filter the dataset for 'Perch' only and rerun the model. 

**Exercise 2:** Using your Perch model, predict the mass of a Perch with length 25 cm. 

**Exercise 3:** Calculate the 95% Confidence Interval and Prediction interval for this 25 cm Perch. Explain why the Prediction Interval is wider.

In [ ]:
# Exercise 1: Filter for Perch and fit model



In [ ]:
# Exercise 2: Predict for 25 cm



In [ ]:
# Exercise 3: Intervals and Interpretation

